# Horse Racing Ranking Model — v2 (Self-Contained)

**Single-file pipeline:** loads data → engineers features → trains 2 models (with/without odds) → compares results → saves artifacts.

## What this notebook does

1. **Loads** `all_tracks_2022_2026.csv` (~951K rows of US thoroughbred races)
2. **Cleans** speed figure outliers from broken source rows
3. **Engineers features** with detailed documentation of *how* each is calculated
4. **Fixes leakage**: jockey/trainer win rates computed from training data ONLY
5. **Adds new features**: class change, distance change, surface change, layoff bins
6. **Trains 2 LightGBM LambdaRank models in parallel:**
   - **Model A** — uses `dollar_odds` as a feature (mirrors public market knowledge)
   - **Model B** — drops `dollar_odds` (forced to find independent signal)
7. **Compares** both models on calibration, NDCG, win accuracy, ROI
8. **Saves** all artifacts to Google Drive

## Why two models?

- Model A piggybacks on public odds. It will look "better" in offline metrics but won't find value bets the public missed.
- Model B has to learn from horse/jockey/trainer/speed signals alone. It will look worse on accuracy but its predictions are *independent* of the market — meaning when it disagrees with the odds, that's a real value signal.

## How to use

1. Upload `all_tracks_2022_2026.csv` and this notebook to Colab
2. Set `CSV_PATH` in Cell 2 to point to your data
3. Set `N_TRIALS` in Cell 8 (5 = quick test, 50 = solid, 100 = full)
4. Run cells in order
5. Compare both models in Cell 12
6. Download artifacts from Cell 13

In [ ]:
# ── Cell 1: Install dependencies ──
!pip install lightgbm optuna scikit-learn scipy joblib shap -q
print("Dependencies installed")

In [ ]:
# ── Cell 2: Configuration & Drive mount ──
# Edit CSV_PATH to point to your dataset.
# OUTPUT_DIR is where model artifacts get saved.

from google.colab import drive
drive.mount('/content/drive')

CSV_PATH = "/content/all_tracks_2022_2026.csv"     # <-- change me
OUTPUT_DIR = "/content/drive/MyDrive/horse_model_v2"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Input:  {CSV_PATH}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
# ── Cell 3: Imports ──
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
import json
import joblib
import warnings
import time
import matplotlib.pyplot as plt
from itertools import permutations
from scipy.special import softmax
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import log_loss, brier_score_loss, ndcg_score

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.INFO)

print("Imports ready")

## Step 1: Load Raw Data

Loads the combined CSV. Each row = one horse in one race.

**Expected columns:** `race_number, race_type, purse, distance, distance_unit, course, surface, track_condition, weather, post_time, win_time, horse_name, breed, weight, age, sex, medication, program_num, post_position, finish, comment, jockey, trainer, owner, last_race_track, last_race_date, last_race_number, last_race_finish, track_code, track_name, race_date, dollar_odds, num_past_starts, num_past_wins, num_past_seconds, num_past_thirds, raw_speed_rating, track_variant, speed_figure, speed_figure_normalized, best_prior_figure, avg_prior_figure, avg_last3_figure, last_figure, num_prior_races, figure_trend, figure_trend_3race, best_surface_figure, avg_surface_figure, best_dist_figure, figure_consistency, peak_vs_recent, furlongs, win_rate, class_level`

**Career stats (`num_past_*`)** are date-aware — they only count races BEFORE the current race date (we computed this in our scraper).

In [ ]:
# ── Cell 4: Load and parse dates ──
print("Loading data...")
t0 = time.time()
df_raw = pd.read_csv(CSV_PATH, low_memory=False)
print(f"  Loaded {len(df_raw):,} rows × {len(df_raw.columns)} columns in {time.time()-t0:.0f}s")

# Parse date columns to actual datetime objects.
# We try multiple formats because the source CSVs use mixed conventions
# (M/D/YYYY from hackathon, YYYY-MM-DD from our scraper).
for date_col in ['race_date', 'last_race_date']:
    if date_col in df_raw.columns:
        df_raw[date_col] = pd.to_datetime(df_raw[date_col], errors='coerce')

# Drop rows missing the basics — without these we can't even build features
df_raw = df_raw.dropna(subset=['horse_name', 'race_date', 'finish', 'race_number', 'track_code'])
df_raw['finish'] = pd.to_numeric(df_raw['finish'], errors='coerce')
df_raw = df_raw.dropna(subset=['finish'])
df_raw['finish'] = df_raw['finish'].astype(int)

print(f"  After dropping incomplete rows: {len(df_raw):,} rows")
print(f"  Date range: {df_raw['race_date'].min().date()} → {df_raw['race_date'].max().date()}")
print(f"  Unique races: {df_raw.groupby(['track_code', 'race_date', 'race_number']).ngroups:,}")

## Step 2: Speed Figure Cleaning

The raw speed figures (`speed_figure_normalized`) and prior-race figures should be in roughly **0–150** for a Beyer-style scale (mean ~75, std ~15). But our source data has some broken rows where:

- `win_time` was 0 (race time missing) → division blew up → figure of -622 or +5047
- Distance was unparseable → bad par-time lookup

**Fix:** clip all speed figure columns to `[0, 150]`. Anything outside that range becomes NaN (LightGBM handles NaN natively, treating it as a separate category).

This is better than dropping the rows entirely because the **other** features (jockey, trainer, post position, etc.) on those rows are still valid.

In [ ]:
# ── Cell 5: Clip speed figure outliers ──
SPEED_FIG_BOUNDS = (0, 150)

speed_fig_columns = [
    'speed_figure_normalized', 'best_prior_figure', 'avg_prior_figure',
    'avg_last3_figure', 'last_figure', 'best_surface_figure',
    'avg_surface_figure', 'best_dist_figure',
]

for col in speed_fig_columns:
    if col not in df_raw.columns:
        continue
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')
    n_outliers = ((df_raw[col] < SPEED_FIG_BOUNDS[0]) | (df_raw[col] > SPEED_FIG_BOUNDS[1])).sum()
    if n_outliers > 0:
        # Replace outliers with NaN — LightGBM will treat as missing
        df_raw.loc[df_raw[col] < SPEED_FIG_BOUNDS[0], col] = np.nan
        df_raw.loc[df_raw[col] > SPEED_FIG_BOUNDS[1], col] = np.nan
        print(f"  Clipped {n_outliers:,} outliers in {col}")

# Sanity check
sf = df_raw['speed_figure_normalized'].dropna()
print(f"\nspeed_figure_normalized after clipping: "
      f"min={sf.min():.1f}, max={sf.max():.1f}, mean={sf.mean():.1f}, std={sf.std():.1f}")

## Step 3: Feature Engineering

This is where we transform raw race data into ML-ready features. Each subsection below explains **exactly how** the feature is computed and why.

### Conventions
- **Numeric features**: NaN is preserved (LightGBM treats it as a separate split)
- **Categorical features**: missing → "Unknown" string
- **Date features**: stored as days/integers, not datetime objects
- **Counts** (`num_past_*`): missing → 0 (first-time starter is meaningfully 0, not unknown)

### Why NaN preservation matters

If a horse has no `last_figure` (first-time starter), that's *information*. Median-imputing it pretends the horse has an average last figure, which is wrong. LightGBM can learn "when last_figure is missing, this horse is a debut runner — apply different rules."

### 3.1: Race ID

A unique identifier for each race so we can group horses by race for ranking.

**Format:** `TRACK_YYYYMMDD_RACENUM` (e.g. `KEE_20250411_3`)

**Why this format:** sorting race IDs lexicographically gives chronological order, which we use for the temporal train/test split.

In [ ]:
# ── Cell 6: Feature Engineering — start with a working copy ──
df = df_raw.copy()

# 3.1: Race ID
df['race_id'] = (
    df['track_code'].astype(str) + '_' +
    df['race_date'].dt.strftime('%Y%m%d') + '_' +
    df['race_number'].astype(str)
)

n_races = df['race_id'].nunique()
print(f"Created {n_races:,} unique race_ids")
print(f"Sample: {df['race_id'].iloc[0]}")

### 3.2: Days Since Last Race (Layoff)

**How calculated:** straight calendar subtraction `(race_date - last_race_date).dt.days`

**NOT distance-based or quality-adjusted** — just raw calendar days. A horse that raced 14 days ago vs 60 days ago has very different fitness levels regardless of where those races were.

**NaN handling:** First-time starters have no `last_race_date`, so this stays NaN. The model learns "missing layoff = debut" which is a meaningful signal.

We also create a **layoff bin** (`layoff_category`):
- `0-7d`: very fresh (just raced last week)
- `8-21d`: normal turnaround 
- `22-45d`: short break
- `46-90d`: layoff
- `90+d`: long layoff (often returns from injury)

These bins help the model find non-linear patterns. A horse coming back from a 6-month layoff is *very* different from one resting 2 weeks, but in raw days that's just "many days vs few days" — categories make the boundary explicit.

In [ ]:
# ── Cell 7a: Layoff features ──
# Vectorized subtraction (fast on 950k rows)
df['days_since_last_race'] = (df['race_date'] - df['last_race_date']).dt.days

# Layoff bins. NaN (first-time starters) stays NaN — model handles it.
df['layoff_category'] = pd.cut(
    df['days_since_last_race'],
    bins=[-1, 7, 21, 45, 90, 9999],
    labels=['very_fresh_0_7d', 'normal_8_21d', 'short_break_22_45d',
            'layoff_46_90d', 'long_layoff_90plus_d']
)

print(f"Layoff distribution:")
print(df['layoff_category'].value_counts().sort_index())
print(f"\nFirst-time starters (NaN layoff): {df['days_since_last_race'].isna().sum():,}")

### 3.3: Race Conditions — Class, Distance, Surface

**`class_level`** (already in source data) — categorical 1-6:
- 1 = Stakes / Graded
- 2 = Allowance
- 3 = Maiden Special Weight
- 4 = Mid Claiming / Maiden Claiming
- 5 = Low Claiming
- 6 = Lowest level

**`distance_furlongs`** — converted to a single unit (furlongs):
- `F` (furlongs): use as-is
- `Y` (yards): divide by 220 (220 yards = 1 furlong)
- `M` (miles): multiply by 8 (8 furlongs = 1 mile)
- Anything else → NaN

**Why furlongs:** standardizes everything to one numeric scale so the model can compare a 6f sprint to a 1.5 mile route directly. Otherwise distance would be an unparseable mix of units.

**`surface`** — categorical: Dirt, Turf, Synthetic, All Weather

**`standard_condition`** — track condition decoded:
- FT → Fast
- GD → Good
- SY → Sloppy
- MY → Muddy
- HY → Heavy
- FM → Firm
- YL → Yielding
- SF → Soft
- (etc.)

These conditions matter because horses run differently on a sloppy track vs a fast track — some specialize in "off" tracks.

In [ ]:
# ── Cell 7b: Race conditions ──

# distance_furlongs may already exist from build_dataset.py — recompute to be safe
if 'distance' in df.columns and 'distance_unit' in df.columns:
    df['distance'] = pd.to_numeric(df['distance'], errors='coerce')
    df['distance_unit'] = df['distance_unit'].astype(str).str.upper()
    
    df['distance_furlongs'] = np.nan
    df.loc[df['distance_unit'] == 'F', 'distance_furlongs'] = df.loc[df['distance_unit'] == 'F', 'distance']
    df.loc[df['distance_unit'] == 'Y', 'distance_furlongs'] = df.loc[df['distance_unit'] == 'Y', 'distance'] / 220
    df.loc[df['distance_unit'] == 'M', 'distance_furlongs'] = df.loc[df['distance_unit'] == 'M', 'distance'] * 8

# Standardize track condition
if 'track_condition' in df.columns:
    track_condition_map = {
        'FT': 'Fast', 'GD': 'Good', 'SY': 'Sloppy', 'MY': 'Muddy',
        'HY': 'Heavy', 'SL': 'Slow', 'FM': 'Firm', 'YL': 'Yielding',
        'SF': 'Soft', 'HD': 'Hard', 'WF': 'Wet Fast', 'FZ': 'Frozen'
    }
    df['standard_condition'] = (
        df['track_condition'].astype(str).str.upper().map(track_condition_map)
        .fillna(df['track_condition'].astype(str))
    )

# Numeric class level
df['class_level'] = pd.to_numeric(df.get('class_level'), errors='coerce')

# Number of horses in each race (for post position normalization)
df['num_horses_in_race'] = df.groupby('race_id')['horse_name'].transform('size')

print(f"distance_furlongs range: {df['distance_furlongs'].min():.1f} - {df['distance_furlongs'].max():.1f}")
print(f"surface values: {df['surface'].value_counts().to_dict()}")
print(f"class_level distribution:\n{df['class_level'].value_counts().sort_index()}")

### 3.4: Post Position Features

The post position is the starting gate number (1 = inside rail, N = outside).

**`post_position_normalized`** — `(post - 1) / (num_horses - 1)`, scaled 0.0 (rail) to 1.0 (outside).

Why normalize? Post 5 in an 8-horse field is middle, but post 5 in a 12-horse field is inside. Raw post position confuses these. Normalized lets the model learn "horses near the rail vs outside" regardless of field size.

**`is_inside_post`** — boolean, 1 if post == 1 (rail). Inside post often has the shortest path on dirt routes.

**`is_outside_post`** — boolean, 1 if post == num_horses (extreme outside). Outside post has to either rush to the front or take wide ground.

In [ ]:
# ── Cell 7c: Post position features ──
df['post_position'] = pd.to_numeric(df.get('post_position'), errors='coerce')

# Normalize: 0.0 = rail, 1.0 = extreme outside
valid_post = (df['num_horses_in_race'] > 1) & df['post_position'].notna()
df['post_position_normalized'] = np.nan
df.loc[valid_post, 'post_position_normalized'] = (
    (df.loc[valid_post, 'post_position'] - 1) /
    (df.loc[valid_post, 'num_horses_in_race'] - 1)
)

df['is_inside_post'] = (df['post_position'] == 1).astype(int)
df['is_outside_post'] = (df['post_position'] == df['num_horses_in_race']).astype(int)

print(f"Post normalized range: {df['post_position_normalized'].min():.2f} - {df['post_position_normalized'].max():.2f}")
print(f"Inside posts: {df['is_inside_post'].sum():,}")
print(f"Outside posts: {df['is_outside_post'].sum():,}")

### 3.5: Career Stats & Win Rate

Already in source data (date-aware from our scraper):
- `num_past_starts` — total prior races BEFORE this one
- `num_past_wins`, `num_past_seconds`, `num_past_thirds`

**Derived:**

**`win_rate`** = `num_past_wins / max(num_past_starts, 1)`

We use `max(starts, 1)` to avoid divide-by-zero. A first-time starter has 0 wins / 1 = 0 win rate. **This is intentional** — we treat unknown rate as 0% rather than NaN because that's the prior expectation (most debut runners don't win).

**`itm_rate`** ("in the money") = `(wins + seconds + thirds) / max(starts, 1)` — how often does this horse hit the board?

**`win_rate_category`** — bins for non-linear modeling:
- Low: 0-10%
- Medium: 10-20%  
- High: 20-30%
- Elite: 30%+

**`experience_category`**:
- Novice: 0-5 starts (still figuring out racing)
- Experienced: 6-15 starts
- Veteran: 16-30 starts
- Elite: 30+ starts

In [ ]:
# ── Cell 7d: Career stats & rates ──
for col in ['num_past_starts', 'num_past_wins', 'num_past_seconds', 'num_past_thirds']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

# Win rate from horse's own prior races (date-aware in source)
df['win_rate'] = df['num_past_wins'] / df['num_past_starts'].clip(lower=1)
df['itm_rate'] = (df['num_past_wins'] + df['num_past_seconds'] + df['num_past_thirds']) / df['num_past_starts'].clip(lower=1)

# Category bins
df['win_rate_category'] = pd.cut(
    df['win_rate'],
    bins=[-0.001, 0.10, 0.20, 0.30, 1.01],
    labels=['Low_0_10pct', 'Medium_10_20pct', 'High_20_30pct', 'Elite_30plus_pct']
)

df['experience_category'] = pd.cut(
    df['num_past_starts'],
    bins=[-1, 5, 15, 30, 9999],
    labels=['Novice_0_5', 'Experienced_6_15', 'Veteran_16_30', 'Elite_30plus']
)

print(f"Win rate distribution:\n{df['win_rate_category'].value_counts()}")
print(f"\nExperience distribution:\n{df['experience_category'].value_counts()}")

### 3.6: Age & Sex

**`age`** — horse age in years at race time. Already in source.

**`age_category`**:
- Young: 2-3 years (still developing)
- Prime: 4-5 years (peak)
- Mature: 6-7 years (experienced but slowing)
- Veteran: 8+ years

**`sex`** — F (filly), M (mare), C (colt), G (gelding), H (horse), R (ridgling)

We bin sex because race conditions are often sex-restricted (fillies-only races, etc.) which the model needs to know.

In [ ]:
# ── Cell 7e: Age category ──
df['age'] = pd.to_numeric(df['age'], errors='coerce')

df['age_category'] = pd.cut(
    df['age'],
    bins=[0, 3, 5, 7, 100],
    labels=['Young_2_3', 'Prime_4_5', 'Mature_6_7', 'Veteran_8plus'],
    right=False
)

print(df['age_category'].value_counts().sort_index())

### 3.7: Limit High-Cardinality Categoricals

**Problem:** there are 1,395 unique jockeys and 3,861 unique trainers. LightGBM can handle high cardinality but it slows down training and many of those have only 1-2 races (no statistical signal).

**Fix:** keep only jockeys with ≥50 starts and trainers with ≥30 starts. Everyone else gets bucketed as `other_jockey` / `other_trainer`.

**Why these thresholds:** 50 starts is enough to estimate a jockey's win rate with reasonable confidence (~7% margin of error). Below that, the rate is just noise.

This is **not** done by win rate (that would leak finish positions). It's done purely by frequency.

In [ ]:
# ── Cell 7f: Bucket rare jockeys/trainers ──
for col, min_starts, other_label in [
    ('jockey', 50, 'other_jockey'),
    ('trainer', 30, 'other_trainer'),
]:
    counts = df[col].value_counts()
    common = set(counts[counts >= min_starts].index)
    n_common = len(common)
    n_total = len(counts)
    df.loc[~df[col].isin(common), col] = other_label
    print(f"{col}: kept {n_common}/{n_total} ({n_common/n_total*100:.0f}%) "
          f"with ≥{min_starts} starts. Bucketed {n_total - n_common} into {other_label}.")

## Step 4: Temporal Split (CRITICAL — before JT rates!)

We split data by date BEFORE computing jockey/trainer win rates. This prevents the leakage that inflated our previous models.

**Split:**
- **Train**: oldest 68% of races (chronological)
- **Calibration**: next 12% (for isotonic regression)
- **Test**: most recent 20% (mimics real deployment)

**Why temporal not random:** A random split would put 2024 races in training and 2023 races in test, letting the model learn from "the future." Real betting only uses the past.

**Why split before JT rates:** If we computed jockey win rates from the full dataset, the model would know future jockey performance for test races. By splitting first, JT rates come ONLY from training data — the model sees the same jockey stats it would have seen in real time.

In [ ]:
# ── Cell 8: Temporal split ──
# Sort by race_date so chronological order is preserved
df = df.sort_values(['race_date', 'race_id']).reset_index(drop=True)

# Get unique races in chronological order
unique_races_sorted = df['race_id'].drop_duplicates().tolist()
n_races_total = len(unique_races_sorted)

train_end = int(n_races_total * 0.68)
cal_end = int(n_races_total * 0.80)

train_race_ids = set(unique_races_sorted[:train_end])
cal_race_ids = set(unique_races_sorted[train_end:cal_end])
test_race_ids = set(unique_races_sorted[cal_end:])

# Boolean masks for splitting
df['split'] = np.where(
    df['race_id'].isin(train_race_ids), 'train',
    np.where(df['race_id'].isin(cal_race_ids), 'cal', 'test')
)

# Date range per split for verification
for split_name in ['train', 'cal', 'test']:
    sub = df[df['split'] == split_name]
    print(f"{split_name:6s}: {len(sub):>7,} rows | "
          f"{sub['race_id'].nunique():>6,} races | "
          f"{sub['race_date'].min().date()} → {sub['race_date'].max().date()}")

### 3.8: Jockey/Trainer Win Rates (LEAK-FREE)

**The leakage we're fixing:** Previously we computed `jockey_trainer_win_rate` from the entire dataset including test races. The model effectively knew "jockey X wins 25% of his races" where that 25% included the very test races we were evaluating. Result: 83% win accuracy in test (impossibly high).

**The fix:** Compute these stats from `df[df['split'] == 'train']` ONLY. Then map those rates onto train/cal/test rows. Test rows get rates that the model could have known *in real time* — no future leakage.

**Stats computed (from training data only):**

| Feature | Formula |
|---------|---------|
| `jockey_win_rate` | (jockey wins in training) / (jockey starts in training) |
| `jockey_itm_rate` | (jockey top-3 finishes in training) / (jockey starts) |
| `trainer_win_rate` | same for trainer |
| `trainer_itm_rate` | same |
| `jockey_trainer_win_rate` | win rate for the specific jockey+trainer combo |
| `jockey_trainer_itm_rate` | same |

**Minimum samples:** Combos with <10 starts in training get NaN (insufficient data → let LightGBM handle as missing).

**For unseen jockeys/trainers/combos** (in cal/test but not in train): NaN. The model learns this means "we don't know this combo's rate."

In [ ]:
# ── Cell 9: Compute jockey/trainer rates from TRAINING DATA ONLY ──
# This is the leakage fix.

train_df = df[df['split'] == 'train'].copy()
train_df['_is_win'] = (train_df['finish'] == 1).astype(int)
train_df['_is_itm'] = (train_df['finish'] <= 3).astype(int)

MIN_STARTS_FOR_RATE = 10

def compute_rate_lookup(group_col):
    """Returns dict {group_value: {'win_rate': x, 'itm_rate': y, 'starts': n}}"""
    grouped = train_df.groupby(group_col, observed=True).agg(
        starts=('_is_win', 'size'),
        wins=('_is_win', 'sum'),
        itm=('_is_itm', 'sum'),
    )
    grouped = grouped[grouped['starts'] >= MIN_STARTS_FOR_RATE]
    grouped['win_rate'] = grouped['wins'] / grouped['starts']
    grouped['itm_rate'] = grouped['itm'] / grouped['starts']
    return grouped[['win_rate', 'itm_rate', 'starts']].to_dict('index')

# Individual jockey stats
print("Computing jockey rates from training data...")
jockey_lookup = compute_rate_lookup('jockey')
print(f"  {len(jockey_lookup)} jockeys with ≥{MIN_STARTS_FOR_RATE} training starts")

# Individual trainer stats
print("Computing trainer rates...")
trainer_lookup = compute_rate_lookup('trainer')
print(f"  {len(trainer_lookup)} trainers with ≥{MIN_STARTS_FOR_RATE} training starts")

# Jockey-trainer combos
print("Computing jockey_trainer combo rates...")
train_df['jockey_trainer'] = train_df['jockey'].astype(str) + '__' + train_df['trainer'].astype(str)
jt_lookup = compute_rate_lookup('jockey_trainer')
print(f"  {len(jt_lookup)} jockey-trainer combos with ≥{MIN_STARTS_FOR_RATE} training starts")

# Apply to all rows (train, cal, test)
df['jockey_trainer'] = df['jockey'].astype(str) + '__' + df['trainer'].astype(str)

df['jockey_win_rate'] = df['jockey'].map(lambda x: jockey_lookup.get(x, {}).get('win_rate', np.nan))
df['jockey_itm_rate'] = df['jockey'].map(lambda x: jockey_lookup.get(x, {}).get('itm_rate', np.nan))
df['trainer_win_rate'] = df['trainer'].map(lambda x: trainer_lookup.get(x, {}).get('win_rate', np.nan))
df['trainer_itm_rate'] = df['trainer'].map(lambda x: trainer_lookup.get(x, {}).get('itm_rate', np.nan))
df['jockey_trainer_win_rate'] = df['jockey_trainer'].map(lambda x: jt_lookup.get(x, {}).get('win_rate', np.nan))
df['jockey_trainer_itm_rate'] = df['jockey_trainer'].map(lambda x: jt_lookup.get(x, {}).get('itm_rate', np.nan))

# Strength bin
df['jt_strength'] = pd.cut(
    df['jockey_trainer_win_rate'],
    bins=[-0.001, 0.08, 0.15, 0.25, 1.01],
    labels=['Weak_0_8pct', 'Average_8_15pct', 'Strong_15_25pct', 'Elite_25plus_pct']
)

# Coverage stats
test_with_jt = df[(df['split'] == 'test') & df['jockey_trainer_win_rate'].notna()]
print(f"\nTest rows with JT combo rate: {len(test_with_jt):,} / {(df['split']=='test').sum():,}")

del train_df  # free memory

## Step 5: Build Feature Matrix

Now we drop columns that aren't features:
- **Identifiers**: `horse_name`, `program_num`, `comment`
- **Display only**: `track_name`, `last_race_track`, `owner`
- **Date columns**: `race_date`, `last_race_date`, `horse_dob` (we extracted what we needed)
- **Speed pipeline intermediate**: `raw_speed_rating`, `track_variant`, `speed_figure` (we have the normalized version)
- **Replaced versions**: `track_condition` (have `standard_condition`), `post_position` (have normalized), `distance` + `distance_unit` (have `distance_furlongs`)
- **Optional**: `dollar_odds` is dropped for Model B but kept for Model A

We DO keep `dollar_odds` in the DataFrame as a separate column for ROI evaluation regardless.

In [ ]:
# ── Cell 10: Build feature matrix ──

# Save things we need separately
race_id_array = df['race_id'].values
finish_array = df['finish'].values
split_array = df['split'].values
dollar_odds_array = pd.to_numeric(df.get('dollar_odds'), errors='coerce').values

# Columns that should NOT be features
NON_FEATURE_COLS = [
    # Identifiers / display
    'horse_name', 'program_num', 'comment', 'race_date', 'last_race_date',
    'last_race_track', 'track_name', 'post_time', 'weather', 'medication', 'owner',
    'horse_dob',
    # Replaced by engineered versions
    'track_condition',     # → standard_condition
    'post_position',       # → post_position_normalized
    'distance',            # → distance_furlongs
    'distance_unit',       # → distance_furlongs
    # Speed pipeline intermediates (not predictive on their own)
    'raw_speed_rating', 'track_variant', 'speed_figure',
    # Fractional times — leak (current race)
    'frac_1', 'frac_2', 'frac_3', 'frac_4',
    # CRITICAL leaks — current race info
    'win_time', 'finish_time', 'final_time',
    'speed_figure_normalized',  # this is the CURRENT race's figure → leak
    # Target and grouping
    'finish', 'race_id', 'split',
]

X_full = df.drop(columns=[c for c in NON_FEATURE_COLS if c in df.columns], errors='ignore')

# Normalize categorical columns
MAX_CATEGORIES_PER_COL = 50
categorical_features = []
for col in X_full.select_dtypes(exclude=np.number).columns:
    X_full[col] = X_full[col].astype('category')
    if 'Unknown' not in X_full[col].cat.categories:
        X_full[col] = X_full[col].cat.add_categories('Unknown')
    X_full[col] = X_full[col].fillna('Unknown')
    
    if X_full[col].nunique() > MAX_CATEGORIES_PER_COL:
        top = X_full[col].value_counts().nlargest(MAX_CATEGORIES_PER_COL).index
        if 'Other' not in X_full[col].cat.categories:
            X_full[col] = X_full[col].cat.add_categories('Other')
        X_full.loc[~X_full[col].isin(top), col] = 'Other'
        X_full[col] = X_full[col].cat.remove_unused_categories()
    
    # Convert to integer codes for LightGBM
    X_full[col] = X_full[col].cat.codes
    categorical_features.append(col)

print(f"Feature matrix: {X_full.shape}")
print(f"Numeric features: {len(X_full.columns) - len(categorical_features)}")
print(f"Categorical features: {len(categorical_features)}")
print(f"\nAll features: {list(X_full.columns)}")

## Step 6: Helper Functions

Group sizes for LambdaRank, NDCG calculation, etc.

In [ ]:
# ── Cell 11: Helper functions ──

def get_group_sizes_ordered(group_array):
    """Vectorized group size extraction. Returns sizes in order of first appearance.
    Used by LambdaRank to know how many horses are in each race group."""
    _, idx, counts = np.unique(group_array, return_index=True, return_counts=True)
    order = np.argsort(idx)
    return counts[order]

def compute_relevance_labels(finish_arr, race_id_arr):
    """Convert finish position → relevance score per race.
    LambdaRank wants higher = better, but finish 1 = winner = best.
    So we invert: relevance = (max_finish_in_race + 1 - finish)."""
    rel = np.zeros(len(finish_arr), dtype=float)
    sizes = get_group_sizes_ordered(race_id_arr)
    idx = 0
    for size in sizes:
        finishes = finish_arr[idx:idx+size]
        max_f = finishes.max()
        rel[idx:idx+size] = max_f + 1 - finishes
        idx += size
    return rel

def compute_race_ndcg_at_k(finish_arr, scores_arr, race_id_arr, k=3):
    """Mean NDCG@k across all races."""
    sizes = get_group_sizes_ordered(race_id_arr)
    ndcgs = []
    idx = 0
    for size in sizes:
        if size < 2:
            idx += size
            continue
        yt = finish_arr[idx:idx+size]
        sc = scores_arr[idx:idx+size]
        max_f = yt.max()
        relevance = (max_f + 1 - yt).reshape(1, -1)
        try:
            ndcgs.append(ndcg_score(relevance, sc.reshape(1, -1), k=k))
        except ValueError:
            pass
        idx += size
    return float(np.mean(ndcgs)) if ndcgs else 0.0

def scores_to_race_probs(scores_arr, race_id_arr):
    """Apply softmax within each race group to convert raw scores → win probabilities."""
    probs = np.zeros_like(scores_arr, dtype=float)
    sizes = get_group_sizes_ordered(race_id_arr)
    idx = 0
    for size in sizes:
        probs[idx:idx+size] = softmax(scores_arr[idx:idx+size])
        idx += size
    return probs

print("Helpers defined")

## Step 7: Train Both Models

We define a single training function and call it twice:
1. **Model A (with odds)** — keeps `dollar_odds` as a feature
2. **Model B (without odds)** — drops `dollar_odds` from features

Both go through the same Optuna search → final training → calibration → evaluation pipeline.

**Configurable:** `N_TRIALS` (5 = quick test, 30 = decent, 100 = thorough)

**Optuna optimizes** NDCG@3 with median pruning (kills bad trials early). Uses 30% subsample of training races for speed (full landscape doesn't change much with more data).

**Final training** uses the best params on the FULL training set with early stopping on the calibration set.

**Calibration** uses isotonic regression to map raw softmax → real win probabilities.

In [ ]:
# ── Cell 12: Training function ──
N_TRIALS = 15  # <-- change me. 5 = quick test, 15 = decent, 50 = thorough, 100 = full

def train_model(model_name, X_data, race_ids, finishes, splits, cat_features, n_trials=N_TRIALS):
    """Full pipeline: Optuna → train → calibrate → evaluate."""
    print(f"\n{'='*70}")
    print(f"TRAINING: {model_name}")
    print(f"{'='*70}")
    print(f"Features: {X_data.shape[1]}")
    
    # Split arrays
    train_mask = splits == 'train'
    cal_mask = splits == 'cal'
    test_mask = splits == 'test'
    
    X_tr = X_data[train_mask].reset_index(drop=True)
    X_cal = X_data[cal_mask].reset_index(drop=True)
    X_te = X_data[test_mask].reset_index(drop=True)
    rids_tr = race_ids[train_mask]
    rids_cal = race_ids[cal_mask]
    rids_te = race_ids[test_mask]
    fin_tr = finishes[train_mask]
    fin_cal = finishes[cal_mask]
    fin_te = finishes[test_mask]
    
    rel_tr = compute_relevance_labels(fin_tr, rids_tr)
    rel_cal = compute_relevance_labels(fin_cal, rids_cal)
    
    print(f"  Train: {len(X_tr):,} rows / {len(np.unique(rids_tr)):,} races")
    print(f"  Cal:   {len(X_cal):,} rows / {len(np.unique(rids_cal)):,} races")
    print(f"  Test:  {len(X_te):,} rows / {len(np.unique(rids_te)):,} races")
    
    # ── Optuna search on 30% subsample ──
    print(f"\nOptuna search ({n_trials} trials, 30% subsample)...")
    unique_train_races = np.unique(rids_tr)
    n_sample = max(500, int(len(unique_train_races) * 0.30))
    rng = np.random.RandomState(42)
    sampled = set(rng.choice(unique_train_races, size=n_sample, replace=False))
    sub_mask = np.array([r in sampled for r in rids_tr])
    
    X_sub = X_tr[sub_mask].reset_index(drop=True)
    rel_sub = rel_tr[sub_mask]
    rids_sub = rids_tr[sub_mask]
    fin_sub = fin_tr[sub_mask]
    
    def objective(trial):
        params = {
            'objective': 'lambdarank',
            'metric': 'ndcg',
            'ndcg_eval_at': [1, 3, 5],
            'verbosity': -1,
            'n_estimators': trial.suggest_int('n_estimators', 300, 1500),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 31, 255),
            'max_depth': trial.suggest_int('max_depth', 5, 15),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 0.9),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 0.95),
            'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
            'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10, log=True),
            'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10, log=True),
            'random_state': 42,
            'n_jobs': -1,
        }
        # Single train/val split for speed (no CV in subsample)
        n_sub = len(unique_train_races)
        n_val = max(100, int(len(sampled) * 0.2))
        sub_arr = np.array(list(sampled))
        rng2 = np.random.RandomState(trial.number)
        val_races = set(rng2.choice(sub_arr, size=n_val, replace=False))
        v_mask = np.array([r in val_races for r in rids_sub])
        
        groups_tr = get_group_sizes_ordered(rids_sub[~v_mask])
        groups_v = get_group_sizes_ordered(rids_sub[v_mask])
        
        ds_tr = lgb.Dataset(X_sub[~v_mask].reset_index(drop=True), label=rel_sub[~v_mask],
                            group=groups_tr, categorical_feature=cat_features)
        ds_v = lgb.Dataset(X_sub[v_mask].reset_index(drop=True), label=rel_sub[v_mask],
                           group=groups_v, categorical_feature=cat_features, reference=ds_tr)
        
        m = lgb.train(params, ds_tr, num_boost_round=params['n_estimators'],
                      valid_sets=[ds_v], callbacks=[lgb.early_stopping(50, verbose=False)])
        scores = m.predict(X_sub[v_mask].reset_index(drop=True))
        return compute_race_ndcg_at_k(fin_sub[v_mask], scores, rids_sub[v_mask], k=3)
    
    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1)
    study = optuna.create_study(direction='maximize', pruner=pruner)
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    
    print(f"\nBest NDCG@3: {study.best_value:.4f}")
    
    # ── Train final model on full train set ──
    print("\nTraining final model on full training set...")
    best_params = {**study.best_params,
                   'objective': 'lambdarank',
                   'metric': 'ndcg',
                   'ndcg_eval_at': [1, 3, 5],
                   'verbosity': -1,
                   'random_state': 42,
                   'n_jobs': -1}
    n_rounds = best_params.pop('n_estimators', 800)
    
    groups_tr = get_group_sizes_ordered(rids_tr)
    groups_cal = get_group_sizes_ordered(rids_cal)
    
    ds_tr = lgb.Dataset(X_tr, label=rel_tr, group=groups_tr,
                        categorical_feature=cat_features)
    ds_cal = lgb.Dataset(X_cal, label=rel_cal, group=groups_cal,
                         categorical_feature=cat_features, reference=ds_tr)
    
    final_model = lgb.train(best_params, ds_tr, num_boost_round=n_rounds,
                             valid_sets=[ds_cal],
                             callbacks=[lgb.early_stopping(200, verbose=False),
                                        lgb.log_evaluation(100)])
    print(f"Best iteration: {final_model.best_iteration}")
    
    # ── Calibration ──
    print("\nCalibrating with isotonic regression...")
    cal_scores = final_model.predict(X_cal)
    cal_raw_probs = scores_to_race_probs(cal_scores, rids_cal)
    y_win_cal = (fin_cal == 1).astype(int)
    
    calibrator = IsotonicRegression(y_min=0.001, y_max=0.999, out_of_bounds='clip')
    calibrator.fit(cal_raw_probs, y_win_cal)
    cal_calibrated = calibrator.predict(cal_raw_probs)
    
    cal_log_loss = log_loss(y_win_cal, cal_calibrated)
    cal_brier = brier_score_loss(y_win_cal, cal_calibrated)
    print(f"  Calibration log-loss: {cal_log_loss:.4f}")
    print(f"  Brier score: {cal_brier:.4f}")
    
    # ── Test set evaluation ──
    print("\nEvaluating on test set...")
    test_scores = final_model.predict(X_te)
    test_raw_probs = scores_to_race_probs(test_scores, rids_te)
    test_calibrated = calibrator.predict(test_raw_probs)
    y_win_te = (fin_te == 1).astype(int)
    
    test_ndcg3 = compute_race_ndcg_at_k(fin_te, test_scores, rids_te, k=3)
    test_ndcg5 = compute_race_ndcg_at_k(fin_te, test_scores, rids_te, k=5)
    test_log_loss = log_loss(y_win_te, test_calibrated)
    
    # Win accuracy: model's top pick = actual winner?
    test_sizes = get_group_sizes_ordered(rids_te)
    win_correct = 0
    n_races_te = len(test_sizes)
    idx = 0
    for size in test_sizes:
        if size < 2:
            idx += size
            continue
        race_scores = test_scores[idx:idx+size]
        race_finish = fin_te[idx:idx+size]
        if np.argmax(race_scores) == np.argmin(race_finish):
            win_correct += 1
        idx += size
    win_acc = win_correct / n_races_te
    
    print(f"  Test NDCG@3: {test_ndcg3:.4f}")
    print(f"  Test NDCG@5: {test_ndcg5:.4f}")
    print(f"  Test log-loss: {test_log_loss:.4f}")
    print(f"  Test win accuracy: {win_acc:.1%} ({win_correct}/{n_races_te})")
    
    # ── ROI simulation ──
    odds_te = dollar_odds_array[test_mask]
    roi_results = {}
    if not np.all(np.isnan(odds_te)):
        print("\nROI simulation (flat $2 win bets):")
        for edge_thresh in [1.10, 1.20, 1.30, 1.50]:
            wagered, returned, n_bets = 0, 0, 0
            idx = 0
            for size in test_sizes:
                race_probs = test_calibrated[idx:idx+size]
                race_probs = race_probs / race_probs.sum() if race_probs.sum() > 0 else race_probs
                race_finish = fin_te[idx:idx+size]
                race_odds = odds_te[idx:idx+size]
                for h in range(size):
                    if np.isnan(race_odds[h]) or race_odds[h] <= 0:
                        continue
                    implied = 1.0 / (race_odds[h] + 1)
                    if race_probs[h] > implied * edge_thresh:
                        wagered += 2
                        n_bets += 1
                        if race_finish[h] == 1:
                            returned += 2 * (race_odds[h] + 1)
                idx += size
            roi = (returned - wagered) / wagered * 100 if wagered > 0 else 0
            print(f"  Edge ≥{edge_thresh:.2f}x: {n_bets:>5} bets, "
                  f"ROI = {roi:+.1f}%, P/L = ${returned-wagered:+.2f}")
            roi_results[f'roi_{edge_thresh}'] = roi
    
    return {
        'name': model_name,
        'model': final_model,
        'calibrator': calibrator,
        'best_params': study.best_params,
        'feature_names': list(X_data.columns),
        'cat_features': cat_features,
        'metrics': {
            'ndcg3': test_ndcg3,
            'ndcg5': test_ndcg5,
            'log_loss': test_log_loss,
            'win_acc': win_acc,
            **roi_results,
        }
    }

In [ ]:
# ── Cell 13: Train Model A (with dollar_odds) ──
X_with_odds = X_full.copy()
result_with_odds = train_model(
    "Model A (WITH dollar_odds)",
    X_with_odds, race_id_array, finish_array, split_array,
    categorical_features, n_trials=N_TRIALS
)

In [ ]:
# ── Cell 14: Train Model B (without dollar_odds) ──
X_no_odds = X_full.drop(columns=['dollar_odds', 'odds_category'], errors='ignore').copy()
no_odds_cats = [c for c in categorical_features if c in X_no_odds.columns]
result_no_odds = train_model(
    "Model B (WITHOUT dollar_odds)",
    X_no_odds, race_id_array, finish_array, split_array,
    no_odds_cats, n_trials=N_TRIALS
)

## Step 8: Compare Both Models

Side-by-side comparison. The interesting comparison is **ROI**, not raw NDCG.

Model A will have higher accuracy because it knows the public's odds. Model B has lower accuracy but its picks are independent of the market — meaning when it disagrees with the favorite, that's a real signal.

In [ ]:
# ── Cell 15: Side-by-side comparison ──
m_a = result_with_odds['metrics']
m_b = result_no_odds['metrics']

print(f"\n{'='*70}")
print(f"{'METRIC':<25} {'Model A (WITH)':>18} {'Model B (NO ODDS)':>18}")
print(f"{'='*70}")
print(f"{'NDCG@3':<25} {m_a['ndcg3']:>18.4f} {m_b['ndcg3']:>18.4f}")
print(f"{'NDCG@5':<25} {m_a['ndcg5']:>18.4f} {m_b['ndcg5']:>18.4f}")
print(f"{'Win Accuracy':<25} {m_a['win_acc']:>17.1%} {m_b['win_acc']:>17.1%}")
print(f"{'Log-loss (lower=better)':<25} {m_a['log_loss']:>18.4f} {m_b['log_loss']:>18.4f}")
for edge in [1.10, 1.20, 1.30, 1.50]:
    key = f'roi_{edge}'
    if key in m_a:
        print(f"{'ROI @'+str(edge)+'x edge':<25} {m_a[key]:>+17.1f}% {m_b[key]:>+17.1f}%")
print(f"{'='*70}")
print()
print("Interpretation:")
print("  - High accuracy alone = doesn't mean profitable")
print("  - ROI @1.20x or higher with +ROI = real edge")
print("  - Model B's edge is independent of public odds")

## Step 9: Save Both Models

Saves all artifacts to Google Drive so `predict.py` can load them.

In [ ]:
# ── Cell 16: Save artifacts ──
def save_model(result, suffix):
    out = lambda f: os.path.join(OUTPUT_DIR, f"ranking_{suffix}_{f}")
    result['model'].save_model(out('model.lgb'))
    joblib.dump(result['calibrator'], out('calibrator.pkl'))
    joblib.dump(result['feature_names'], out('feature_names.pkl'))
    joblib.dump(result['cat_features'], out('cat_features.pkl'))
    joblib.dump(result['best_params'], out('best_params.pkl'))
    with open(out('metrics.json'), 'w') as f:
        json.dump(result['metrics'], f, indent=2)
    print(f"Saved Model {suffix}: {out('model.lgb')}")

save_model(result_with_odds, "with_odds")
save_model(result_no_odds, "no_odds")

# Save jockey/trainer lookups (used by both predict.py and inspection)
jt_lookups = {
    'jockey': {k: v for k, v in jockey_lookup.items()},
    'trainer': {k: v for k, v in trainer_lookup.items()},
    'jockey_trainer': {k: v for k, v in jt_lookup.items()},
}
with open(os.path.join(OUTPUT_DIR, 'ranking_jt_lookup.json'), 'w') as f:
    json.dump(jt_lookups, f)
print(f"Saved jockey/trainer lookups: {OUTPUT_DIR}/ranking_jt_lookup.json")
print(f"\nAll files in {OUTPUT_DIR}/")
!ls -lh {OUTPUT_DIR}

## Step 10: Feature Importance (SHAP)

Quick check on what features the model actually uses. Run for whichever model you want to inspect.

In [ ]:
# ── Cell 17: SHAP analysis ──
import shap

# Pick which model to analyze
INSPECT = result_no_odds  # or result_with_odds

# Sample test data for speed
test_mask = split_array == 'test'
if INSPECT['name'].startswith('Model A'):
    X_inspect = X_with_odds[test_mask].reset_index(drop=True)
else:
    X_inspect = X_no_odds[test_mask].reset_index(drop=True)

if len(X_inspect) > 5000:
    X_inspect = X_inspect.sample(5000, random_state=42)

print(f"Computing SHAP values for {INSPECT['name']} on {len(X_inspect)} samples...")
explainer = shap.TreeExplainer(INSPECT['model'])
shap_values = explainer.shap_values(X_inspect)

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_inspect, feature_names=INSPECT['feature_names'],
                  max_display=25, show=False)
plt.title(f"SHAP — {INSPECT['name']}")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f"shap_{INSPECT['name'].split()[1].lower()}.png"), dpi=150, bbox_inches='tight')
plt.show()